# Evaluasi RAGAS - Chatbot Mitigasi Bencana BPBD Kabupaten Bogor

In [ ]:
# SEL 1: Import dan Koneksi
import os
import re
from dotenv import load_dotenv
from langchain_neo4j import Neo4jGraph
from langchain_ollama import OllamaLLM
from openai import OpenAI
from ragas.llms import llm_factory

load_dotenv(dotenv_path='.env', override=True)

# Koneksi Neo4j
graph = Neo4jGraph(
    url=os.getenv('NEO4J_URI'),
    username=os.getenv('NEO4J_USERNAME'),
    password=os.getenv('NEO4J_PASSWORD'),
    database=os.getenv('NEO4J_DATABASE')
)

# LLM Ollama
llm_ollama = OllamaLLM(model='gemma3:1b')

# OpenAI untuk RAGAS judge
openai_client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
ragas_llm = llm_factory(
    'gpt-4.1-nano',
    client=openai_client,
    temperature=0
)

print('✅ Koneksi berhasil!')

In [ ]:
# SEL 2: Keywords dan Fungsi cari_konteks_graph
KEYWORDS_SPESIFIK = {
    'angin': 'Angin_Kencang.pdf',
    'banjir': 'Banjir.pdf',
    'gempa': 'Gempa_Bumi.pdf',
    'longsor': 'Tanah_Longsor.pdf',
    'erupsi': 'Erupsi_Gunung_Api.pdf',
    'gunung': 'Erupsi_Gunung_Api.pdf',
    'karhutla': 'Karhutla.pdf',
    'kebakaran': 'Karhutla.pdf',
    'kekeringan': 'Kekeringan.pdf',
    'pergeseran': 'Pergeseran_Tanah.pdf',
}

KEYWORDS_UMUM = ['mitigasi', 'bencana', 'darurat', 'evakuasi']

def cari_konteks_graph(pertanyaan):
    pertanyaan_bersih = re.sub(r'[^\w\s]', ' ', pertanyaan.lower())
    words = pertanyaan_bersih.split()
    
    keywords_spesifik = [k for k in words if k in KEYWORDS_SPESIFIK]
    keywords_umum = [k for k in words if k in KEYWORDS_UMUM]
    keywords = keywords_spesifik if keywords_spesifik else keywords_umum
    
    if not keywords:
        semua_keywords_dikenal = list(KEYWORDS_SPESIFIK.keys()) + KEYWORDS_UMUM
        keywords = [w for w in words if any(k in w for k in semua_keywords_dikenal)]
        if not keywords:
            return 'Tidak ada konteks ditemukan.'

    kondisi = ' OR '.join([f"toLower(e.nama) CONTAINS '{k}'" for k in keywords])

    if keywords_spesifik:
        dokumen_target = [KEYWORDS_SPESIFIK[k] for k in keywords_spesifik]
        kondisi_dokumen = ' OR '.join([f"d.nama = '{d}'" for d in dokumen_target])
    else:
        kondisi_dokumen = ' OR '.join([f"toLower(e.nama) CONTAINS '{k}'" for k in keywords])

    hasil_relasi = graph.query(f"""
        MATCH (e:Entitas)
        WHERE {kondisi}
        OPTIONAL MATCH (e)-[r]->(target:Entitas)
        RETURN e.nama AS entitas,
               collect(DISTINCT type(r) + ' -> ' + target.nama) AS relasi_keluar
    """)

    konteks_lines = []
    for item in hasil_relasi:
        entitas = item['entitas']
        keluar = [r for r in item['relasi_keluar'] if r and '-> None' not in r]
        if keluar:
            targets = [r.split('-> ')[1] for r in keluar if '-> ' in r]
            if targets:
                konteks_lines.append(f'• {entitas} berkaitan dengan: {", ".join(targets[:5])}')

    return '\n'.join(konteks_lines) if konteks_lines else 'Tidak ada konteks ditemukan.'

print('✅ Fungsi siap!')

In [ ]:
# SEL 3: Kumpulkan Jawaban Chatbot
from datasets import Dataset

pertanyaan_list = [
    'Apa yang harus dilakukan saat terjadi banjir?',
    'Bagaimana cara mitigasi longsor?',
    'Apa langkah evakuasi saat gempa bumi?',
    'Bagaimana cara mempersiapkan diri sebelum bencana angin kencang?',
    'Apa yang dilakukan pasca bencana erupsi gunung api?'
]

def dapatkan_jawaban(pertanyaan):
    konteks = cari_konteks_graph(pertanyaan)
    prompt = f"""
Kamu adalah asisten mitigasi bencana BPBD Kabupaten Bogor.
Informasi dari basis data:
---
{konteks}
---
Pertanyaan: {pertanyaan}
Jawab singkat, padat, dan jelas per fase (Pra, Saat, Pasca Bencana). Maksimal 3 poin per fase.
"""
    return llm_ollama.invoke(prompt), konteks

print('🔄 Mengumpulkan jawaban chatbot...')
data = {'question': [], 'answer': [], 'contexts': [], 'reference': []}

for pertanyaan in pertanyaan_list:
    print(f'  ❓ {pertanyaan}')
    jawaban, konteks = dapatkan_jawaban(pertanyaan)
    data['question'].append(pertanyaan)
    data['answer'].append(jawaban)
    data['contexts'].append([konteks])
    data['reference'].append(konteks)
    print(f'  ✅ Jawaban terkumpul')

dataset = Dataset.from_dict(data)
print('\n✅ Dataset siap dievaluasi!')
print(f'Total pertanyaan: {len(data["question"])}')

In [ ]:
# SEL 4: Evaluasi RAGAS
from ragas import evaluate
from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision

print('📊 Mengevaluasi dengan RAGAS...')
metrics = [
    Faithfulness(llm=ragas_llm),
    AnswerRelevancy(llm=ragas_llm),
    ContextPrecision(llm=ragas_llm)
]

hasil = evaluate(dataset=dataset, metrics=metrics)
print('✅ Evaluasi selesai!')

In [ ]:
# SEL 5: Tampilkan dan Simpan Hasil
import pandas as pd

df = hasil.to_pandas()

print('\n✅ Hasil Evaluasi RAGAS:')
print(f'  Faithfulness      : {df["faithfulness"].mean():.4f}')
print(f'  Answer Relevancy  : {df["answer_relevancy"].mean():.4f}')
print(f'  Context Precision : {df["context_precision"].mean():.4f}')

print('\n📋 Detail per pertanyaan:')
print(df[['question', 'faithfulness', 'answer_relevancy', 'context_precision']].to_string(index=False))

df.to_csv('hasil_evaluasi.csv', index=False)
print('\n💾 Hasil disimpan ke hasil_evaluasi.csv')